In [8]:
import requests
import pandas as pd
import time

cities = ["London", "New York", "Tokyo", "Cairo", "Sydney"]
weather_data = []

for city in cities:
    # Use wttr.in, an open weather service that returns JSON
    url = f"https://wttr.in/{city}?format=j1"
    
    try:
        response = requests.get(url)
        # Parse the JSON response
        data = response.json()
        
        # Extract current condition metrics
        current_condition = data['current_condition'][0]
        
        temp = int(current_condition['temp_C'])
        humidity = int(current_condition['humidity'])
        wind = int(current_condition['windspeedKmph'])
        
        weather_data.append({
            'City': city, 
            'Temperature': temp, 
            'Humidity (%)': humidity, 
            'Wind Speed': wind
        })
        print(f"Successfully retrieved data for {city}")
        
    except Exception as e:
        print(f"Could not retrieve data for {city}. Error: {e}")
        
    # Brief pause to be polite to the API
    time.sleep(1)

# Store the data in a CSV file
df = pd.DataFrame(weather_data)
df.to_csv('weather_data.csv', index=False)
print("\nData successfully saved to weather_data.csv.")

Successfully retrieved data for London
Successfully retrieved data for New York
Successfully retrieved data for Tokyo
Successfully retrieved data for Cairo
Successfully retrieved data for Sydney

Data successfully saved to weather_data.csv.


In [9]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px

# Load the scraped dataset
df = pd.read_csv('weather_data.csv')
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Global Weather Analytics Dashboard"),
    
    # 1 Filter (Dropdown)
    dcc.Dropdown(
        id='city-filter', 
        options=[{'label': city, 'value': city} for city in df['City']], 
        multi=True, 
        placeholder="Select cities to compare..."
    ),
    
    # 4 KPI Cards
    html.Div(id='kpi-cards', style={'display': 'flex', 'gap': '20px', 'padding': '20px 0'}),
    
    # 2 Charts
    dcc.Graph(id='temp-chart'),
    dcc.Graph(id='humidity-chart')
])

@app.callback(
    [Output('kpi-cards', 'children'), Output('temp-chart', 'figure'), Output('humidity-chart', 'figure')],
    [Input('city-filter', 'value')]
)
def update_dashboard(selected_cities):
    filtered_df = df[df['City'].isin(selected_cities)] if selected_cities else df
    
    # Generate KPI Cards dynamically based on filtered data
    kpis = [
        html.Div([html.H3("Avg Temp"), html.P(f"{filtered_df['Temperature'].mean():.1f}°")]),
        html.Div([html.H3("Max Temp"), html.P(f"{filtered_df['Temperature'].max()}°")]),
        html.Div([html.H3("Avg Humidity"), html.P(f"{filtered_df['Humidity (%)'].mean():.1f}%")]),
        html.Div([html.H3("Max Wind Speed"), html.P(f"{filtered_df['Wind Speed'].max()} mph")])
    ]
    
    # Generate 2 Interactive Charts
    fig_temp = px.bar(filtered_df, x='City', y='Temperature', title="Temperature Comparison Across Cities", color='City')
    fig_humidity = px.scatter(filtered_df, x='City', y='Humidity (%)', size='Wind Speed', title="Humidity vs Wind Speed (Bubble Size)", color='City')
    
    return kpis, fig_temp, fig_humidity

if __name__ == '__main__':
    # Launches external link rather than inline notebook display
    app.run(debug=True, jupyter_mode="external")

Dash app running on http://127.0.0.1:8050/
